# Анализ робастности моделей DDIB

Этот ноутбук предназначен для анализа результатов валидации робастности лучших конфигураций моделей.

## Метрики робастности

- **mCE (mean Corruption Error)**: Средняя ошибка на искажениях CIFAR-10-C, нормализованная относительно референсной модели
- **mAA (mean Adversarial Accuracy)**: Средняя точность на adversarial примерах (PGD-атака)
- **Robustness Gap**: Разница между clean accuracy и adversarial accuracy

## Дatasets для валидации

- **CIFAR-10-C**: 15 типов искажений × 5 уровней серьезности
- **Adversarial examples**: PGD-атака с ε=8/255, α=2/255, 10 итераций

In [ ]:
import json
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
warnings.filterwarnings('ignore')

# Fixed colormap
ROBUSTNESS_CMAP = 'viridis'

In [ ]:
def get_notebook_dir():
    """Get the directory containing the current notebook or script."""
    try:
        return Path(__file__).parent.resolve()
    except NameError:
        return Path.cwd().resolve()

# Paths
PROJECT_ROOT = get_notebook_dir().parent
RESULTS_PATH = PROJECT_ROOT / 'results' / 'robustness'
OUTPUT_DIR = PROJECT_ROOT / 'reports' / 'robustness' / 'analysis'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Loading results from: {RESULTS_PATH}')
print(f'Saving analysis to: {OUTPUT_DIR}')

## Загрузка результатов

In [ ]:
def load_robustness_results(results_path: Path) -> pd.DataFrame:
    """
    Load robustness evaluation results from JSON files.
    
    Args:
        results_path: Directory containing model result folders
        
    Returns:
        DataFrame with all results
    """
    results = []
    
    for model_dir in results_path.iterdir():
        if not model_dir.is_dir():
            continue
        
        summary_path = model_dir / 'robustness_summary.json'
        if not summary_path.exists():
            continue
        
        with open(summary_path, 'r') as f:
            data = json.load(f)
        
        data['model_name'] = model_dir.name
        results.append(data)
    
    return pd.DataFrame(results)

# Try to load results
if RESULTS_PATH.exists():
    df = load_robustness_results(RESULTS_PATH)
    print(f'Loaded {len(df)} model results')
    print(f'\nColumns: {list(df.columns)}')
else:
    print('No results found. Run validation first:')
    print('  python -m src.experiments.robustness.validate --evaluate-all')
    df = pd.DataFrame()

## Основные метрики робастности

In [ ]:
if len(df) > 0:
    # Summary statistics
    print('ROBUSTNESS METRICS SUMMARY')
    print('=' * 60)
    
    metrics_cols = ['clean_accuracy', 'mCE', 'mAA', 'adversarial_accuracy', 'robustness_gap']
    available_cols = [c for c in metrics_cols if c in df.columns]
    
    summary = df[available_cols].describe().loc[['mean', 'std', 'min', 'max']]
    print(summary.round(4))

In [ ]:
if len(df) > 0:
    # Comparison table
    print('\nMODEL COMPARISON')
    print('=' * 80)
    
    display_cols = ['model_name', 'clean_accuracy', 'mCE', 'mAA', 'adversarial_accuracy', 'robustness_gap']
    available_display = [c for c in display_cols if c in df.columns]
    
    comparison = df[available_display].copy()
    comparison = comparison.sort_values('mCE', ascending=True)
    
    # Format for display
    for col in comparison.columns:
        if col != 'model_name' and comparison[col].dtype in ['float64', 'int64']:
            comparison[col] = comparison[col].apply(lambda x: f'{x:.4f}' if pd.notna(x) else 'N/A')
    
    print(comparison.to_string(index=False))

## Визуализация результатов

In [ ]:
if len(df) > 0 and 'mCE' in df.columns and 'clean_accuracy' in df.columns:
    # Plot 1: mCE vs Clean Accuracy
    fig, ax = plt.subplots(figsize=(10, 6))
    
    scatter = ax.scatter(
        df['clean_accuracy'],
        df['mCE'],
        s=100,
        c=df.get('mAA', np.ones(len(df))),
        cmap=ROBUSTNESS_CMAP,
        alpha=0.7,
        edgecolors='black',
        linewidth=1,
    )
    
    # Add model labels
    for i, row in df.iterrows():
        ax.annotate(
            row['model_name'][:15],
            (row['clean_accuracy'], row['mCE']),
            xytext=(5, 5),
            textcoords='offset points',
            fontsize=8,
        )
    
    ax.set_xlabel('Clean Accuracy')
    ax.set_ylabel('Mean Corruption Error (mCE)')
    ax.set_title('Robustness: mCE vs Clean Accuracy')
    
    cbar = plt.colorbar(scatter)
    cbar.set_label('Mean Adversarial Accuracy (mAA)')
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'mce_vs_clean_accuracy.png', dpi=150)
    print(f'Saved: mce_vs_clean_accuracy.png')
    plt.show()

In [ ]:
if len(df) > 0 and 'robustness_gap' in df.columns:
    # Plot 2: Robustness Gap comparison
    fig, ax = plt.subplots(figsize=(10, 6))
    
    models = df['model_name'].values
    gaps = df['robustness_gap'].values
    clean_acc = df['clean_accuracy'].values
    adv_acc = df.get('adversarial_accuracy', pd.Series([0]*len(df))).values
    
    x = np.arange(len(models))
    width = 0.35
    
    bars1 = ax.bar(x - width/2, clean_acc, width, label='Clean Accuracy', color='steelblue')
    bars2 = ax.bar(x + width/2, adv_acc, width, label='Adversarial Accuracy', color='coral')
    
    ax.set_ylabel('Accuracy')
    ax.set_xlabel('Model')
    ax.set_title('Clean vs Adversarial Accuracy by Model')
    ax.set_xticks(x)
    ax.set_xticklabels([m[:12] for m in models], rotation=45, ha='right')
    ax.legend()
    ax.set_ylim(0, 1)
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'clean_vs_adversarial.png', dpi=150)
    print(f'Saved: clean_vs_adversarial.png')
    plt.show()

In [ ]:
if len(df) > 0:
    # Plot 3: Corruption Error by type (if available)
    ce_cols = [c for c in df.columns if c.startswith('CE_')]
    
    if ce_cols:
        fig, ax = plt.subplots(figsize=(12, 8))
        
        # Extract corruption names
        corruption_names = [c.replace('CE_', '') for c in ce_cols]
        
        # Create dataframe for plotting
        ce_data = df[ce_cols].copy()
        ce_data.columns = corruption_names
        ce_data.index = df['model_name']
        
        # Heatmap
        sns.heatmap(ce_data.T, annot=True, fmt='.3f', cmap=ROBUSTNESS_CMAP, ax=ax,
                   cbar_kws={'label': 'Corruption Error'})
        
        ax.set_xlabel('Model')
        ax.set_ylabel('Corruption Type')
        ax.set_title('Corruption Error by Type and Model')
        
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / 'corruption_error_heatmap.png', dpi=150)
        print(f'Saved: corruption_error_heatmap.png')
        plt.show()

In [ ]:
if len(df) > 0:
    # Plot 4: Severity analysis
    # Load detailed results if available
    detailed_results = []
    
    for model_dir in RESULTS_PATH.iterdir():
        if not model_dir.is_dir():
            continue
        
        full_path = model_dir / 'robustness_results.json'
        if not full_path.exists():
            continue
        
        with open(full_path, 'r') as f:
            data = json.load(f)
        
        if 'corruption_accuracies' in data:
            for corruption_type, severities in data['corruption_accuracies'].items():
                for severity, accuracy in severities.items():
                    detailed_results.append({
                        'model_name': model_dir.name,
                        'corruption_type': corruption_type,
                        'severity': int(severity),
                        'accuracy': accuracy,
                    })
    
    if detailed_results:
        detailed_df = pd.DataFrame(detailed_results)
        
        # Plot accuracy by severity
        fig, ax = plt.subplots(figsize=(10, 6))
        
        severity_acc = detailed_df.groupby(['model_name', 'severity'])['accuracy'].mean().reset_index()
        
        for model in detailed_df['model_name'].unique():
            model_data = severity_df[severity_df['model_name'] == model]
            ax.plot(model_data['severity'], model_data['accuracy'], marker='o', label=model[:15])
        
        ax.set_xlabel('Severity Level')
        ax.set_ylabel('Accuracy')
        ax.set_title('Accuracy vs Corruption Severity')
        ax.set_xticks([1, 2, 3, 4, 5])
        ax.legend(loc='lower left')
        
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / 'accuracy_by_severity.png', dpi=150)
        print(f'Saved: accuracy_by_severity.png')
        plt.show()

## Анализ по типам искажений

In [ ]:
if len(df) > 0:
    # Group corruptions by type
    noise_corruptions = ['gaussian_noise', 'shot_noise', 'impulse_noise']
    blur_corruptions = ['defocus_blur', 'glass_blur', 'motion_blur', 'zoom_blur']
    weather_corruptions = ['snow', 'frost', 'fog']
    digital_corruptions = ['brightness', 'contrast', 'elastic_transform', 'pixelate', 'jpeg_compression']
    
    corruption_groups = {
        'Noise': noise_corruptions,
        'Blur': blur_corruptions,
        'Weather': weather_corruptions,
        'Digital': digital_corruptions,
    }
    
    # Calculate average error by group
    group_errors = {}
    
    for model_name in df['model_name'].unique():
        model_row = df[df['model_name'] == model_name].iloc[0]
        group_errors[model_name] = {}
        
        for group_name, corruptions in corruption_groups.items():
            ce_cols = [f'CE_{c}' for c in corruptions if f'CE_{c}' in model_row.index]
            if ce_cols:
                group_errors[model_name][group_name] = model_row[ce_cols].mean()
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 6))
    
    group_df = pd.DataFrame(group_errors).T
    group_df.plot(kind='bar', ax=ax, colormap=ROBUSTNESS_CMAP)
    
    ax.set_xlabel('Model')
    ax.set_ylabel('Average Corruption Error')
    ax.set_title('Corruption Error by Category')
    ax.legend(title='Category')
    plt.xticks(rotation=45, ha='right')
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'error_by_category.png', dpi=150)
    print(f'Saved: error_by_category.png')
    plt.show()

## Финальный отчёт

In [ ]:
if len(df) > 0:
    print('=' * 60)
    print('ROBUSTNESS EVALUATION SUMMARY')
    print('=' * 60)
    
    # Best model by mCE (lower is better)
    if 'mCE' in df.columns:
        best_mce_idx = df['mCE'].idxmin()
        best_mce_model = df.loc[best_mce_idx]
        print(f"\nBest mCE: {best_mce_model['model_name']} (mCE = {best_mce_model['mCE']:.4f})")
    
    # Best model by mAA (higher is better)
    if 'mAA' in df.columns:
        best_maa_idx = df['mAA'].idxmax()
        best_maa_model = df.loc[best_maa_idx]
        print(f"Best mAA: {best_maa_model['model_name']} (mAA = {best_maa_model['mAA']:.4f})")
    
    # Best clean accuracy
    if 'clean_accuracy' in df.columns:
        best_clean_idx = df['clean_accuracy'].idxmax()
        best_clean_model = df.loc[best_clean_idx]
        print(f"Best Clean Accuracy: {best_clean_model['model_name']} (Acc = {best_clean_model['clean_accuracy']:.4f})")
    
    # Most robust (smallest gap)
    if 'robustness_gap' in df.columns:
        best_gap_idx = df['robustness_gap'].idxmin()
        best_gap_model = df.loc[best_gap_idx]
        print(f"Most Robust (smallest gap): {best_gap_model['model_name']} (Gap = {best_gap_model['robustness_gap']:.4f})")
    
    print('\n' + '=' * 60)

In [ ]:
# Save summary report
if len(df) > 0:
    report_path = OUTPUT_DIR / 'robustness_report.txt'
    
    with open(report_path, 'w') as f:
        f.write('ROBUSTNESS EVALUATION REPORT\n')
        f.write('=' * 60 + '\n\n')
        
        f.write(f'Total models evaluated: {len(df)}\n\n')
        
        # Summary statistics
        metrics_cols = ['clean_accuracy', 'mCE', 'mAA', 'adversarial_accuracy']
        available_cols = [c for c in metrics_cols if c in df.columns]
        
        for col in available_cols:
            f.write(f'{col}:\n')
            f.write(f'  Mean: {df[col].mean():.4f}\n')
            f.write(f'  Std:  {df[col].std():.4f}\n')
            f.write(f'  Min:  {df[col].min():.4f}\n')
            f.write(f'  Max:  {df[col].max():.4f}\n\n')
        
        # Best models
        f.write('BEST MODELS\n')
        f.write('-' * 40 + '\n')
        
        if 'mCE' in df.columns:
            best = df.loc[df['mCE'].idxmin()]
            f.write(f"Best mCE: {best['model_name']} ({best['mCE']:.4f})\n")
        
        if 'mAA' in df.columns:
            best = df.loc[df['mAA'].idxmax()]
            f.write(f"Best mAA: {best['model_name']} ({best['mAA']:.4f})\n")
        
        if 'clean_accuracy' in df.columns:
            best = df.loc[df['clean_accuracy'].idxmax()]
            f.write(f"Best Clean Acc: {best['model_name']} ({best['clean_accuracy']:.4f})\n")
    
    print(f'Report saved to: {report_path}')